# Merged Semantic Scholar profiles: what the batch call returns

Claim to check:

1. `POST /author/batch` returns **null** for an id that Semantic Scholar merged into another profile (it does not say which one).
2. `GET /author/{old_id}` follows the merge and returns the **new** `authorId`.
3. The new id does come back in the batch call.

Test ids:
- **merged**: old ids of confirmed researchers (`factuality_author = True`) whose individual call gave another id (`results/summary/openalex_cascade/ss_papers_parts/merged_*.parquet`).
- **control**: ids that the batch call did return, with papers (`ss_papers_all/ss_papers_parts/batch_*.parquet`).

Only ids and counts are shown (no names).

In [1]:
import configparser
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd

API = "https://api.semanticscholar.org/graph/v1"
RESULTS = Path("../../../../results/summary")
N = 10   # ids per group


def api_key():
    """The file that [semantic_scholar].s2_api_key in config.ini points to."""
    cfg = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "config.ini").exists():
            cfg.read(d / "config.ini")
            p = Path(cfg.get("semantic_scholar", "s2_api_key", fallback="")).expanduser()
            return (p if p.is_absolute() else d / p).read_text().strip()


KEY = api_key()


def call(path, body=None):
    """One call (1 per second, the key's limit), retried on 429. Returns (HTTP status, answer)."""
    headers = {"Content-Type": "application/json", "x-api-key": KEY}
    for attempt in range(10):
        time.sleep(1.1)
        req = urllib.request.Request(f"{API}{path}", headers=headers, method="POST" if body else "GET",
                                     data=json.dumps(body).encode() if body else None)
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.status, json.load(r)
        except urllib.error.HTTPError as e:
            if e.code != 429:
                return e.code, None
        time.sleep(2 ** attempt)

## The test ids

In [2]:
parts = RESULTS / "openalex_cascade/ss_papers_parts"
merged = pd.concat([pd.read_parquet(f) for f in sorted(parts.glob("merged_*.parquet"))[:300]])
merged = merged.dropna(subset=["current_id"])
merged = merged[merged["current_id"].astype("int64") != merged["researcher_id"].astype("int64")]
merged = merged.sample(N, random_state=0).reset_index(drop=True)
merged["researcher_id"] = merged["researcher_id"].astype("int64").astype(str)
merged["current_id"] = merged["current_id"].astype("int64").astype(str)

control = pd.read_parquet(RESULTS / "ss_papers_all/ss_papers_parts/batch_00000.parquet", columns=["researcher_id"])
control = control.drop_duplicates().sample(N, random_state=0)["researcher_id"].astype("int64").astype(str).tolist()
print(len(merged), "merged ids,", len(control), "control ids")

10 merged ids, 10 control ids


## 1. Batch call with old (merged) ids and control ids, in the same request

In [3]:
ids = merged["researcher_id"].tolist() + control
status, answer = call("/author/batch?fields=authorId,paperCount", {"ids": ids})
batch = pd.DataFrame({
    "group": ["merged"] * N + ["control"] * N,
    "asked_id": ids,
    "answer_is_null": [a is None for a in answer],
    "returned_id": [a["authorId"] if a else None for a in answer],
    "paperCount": [a["paperCount"] if a else None for a in answer],
})
print("HTTP", status, "| answers:", len(answer), "for", len(ids), "ids (same order)")
batch

HTTP 200 | answers: 20 for 20 ids (same order)


,group,asked_id,answer_is_null,returned_id,paperCount
0,merged,1895832,True,None,NaN
1,merged,1820663,True,None,NaN
2,merged,2104331,True,None,NaN
3,merged,1688254,True,None,NaN
4,merged,1970960,True,None,NaN
5,merged,1983936,True,None,NaN
6,merged,1739851,True,None,NaN
7,merged,2106689,True,None,NaN
8,merged,1928207,True,None,NaN
9,merged,1916400,True,None,NaN


In [4]:
batch.groupby("group")["answer_is_null"].agg(["sum", "count"]).rename(columns={"sum": "null", "count": "asked"})

,null,asked
group,,
control,0,10
merged,10,10


## 2. Individual call with each old id: does it follow the merge?

In [5]:
rows = []
for old, saved_new in zip(merged["researcher_id"], merged["current_id"]):
    status, a = call(f"/author/{old}?fields=authorId,paperCount")
    rows.append({"old_id": old, "http": status, "returned_id": a and a["authorId"],
                 "returned_id != old_id": bool(a) and a["authorId"] != old,
                 "returned_id == saved current_id": bool(a) and a["authorId"] == saved_new,
                 "paperCount": a and a["paperCount"]})
single = pd.DataFrame(rows)
single

,old_id,http,returned_id,returned_id != old_id,returned_id == saved current_id,paperCount
0,1895832,200,73230092,True,True,123
1,1820663,200,143944173,True,True,192
2,2104331,200,50098623,True,True,215
3,1688254,200,2384140520,True,True,3
4,1970960,200,2111301930,True,True,3
5,1983936,200,47755780,True,True,87
6,1739851,200,145306271,True,True,283
7,2106689,200,121451396,True,True,2
8,1928207,200,52058983,True,True,28
9,1916400,200,144574835,True,True,593


## 3. Batch call with the new ids

In [6]:
status, answer = call("/author/batch?fields=authorId,paperCount", {"ids": merged["current_id"].tolist()})
new = pd.DataFrame({"new_id": merged["current_id"], "answer_is_null": [a is None for a in answer],
                    "paperCount": [a["paperCount"] if a else None for a in answer]})
print("HTTP", status)
new

HTTP 200


,new_id,answer_is_null,paperCount
0,73230092,False,123
1,143944173,False,192
2,50098623,False,215
3,2384140520,False,3
4,2111301930,False,3
5,47755780,False,87
6,145306271,False,283
7,121451396,False,2
8,52058983,False,28
9,144574835,False,593


## Summary

In [7]:
pd.DataFrame({
    "check": ["batch, old (merged) ids → null",
              "batch, control ids → not null",
              "individual call, old id → another id",
              "  ... the same id saved by 1_ss_papers.py",
              "batch, new ids → not null"],
    "yes": [batch.query("group == 'merged'")["answer_is_null"].sum(),
            (~batch.query("group == 'control'")["answer_is_null"]).sum(),
            single["returned_id != old_id"].sum(),
            single["returned_id == saved current_id"].sum(),
            (~new["answer_is_null"]).sum()],
    "of": [N, N, N, N, N],
})

,check,yes,of
0,"batch, old (merged) ids → null",10,10
1,"batch, control ids → not null",10,10
2,"individual call, old id → another id",10,10
3,... the same id saved by 1_ss_papers.py,10,10
4,"batch, new ids → not null",10,10
